# Exploring the Forecast Arena record

The open data of [Forecast Arena](https://jashd12.github.io/forecast-arena/data.html) is four Parquet
files: `questions`, `forecasts`, `resolutions` and `skips` (column meanings: the data dictionary on
the site's *Open data & SQL* page). This notebook reads them with **DuckDB** (SQL directly on files)
and **pandas**, recomputes the Brier score per forecaster, looks at calibration and at why
forecasts were skipped.

Download the files from the site's data page into a folder and set `ARENA_DATA_DIR` to it (default:
`data`). In CI this notebook runs on the offline end-to-end export, which is **fixture data, not the
live record**.

These are exploratory numbers: the headline comparison with 95% intervals is on the leaderboard.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

DATA = Path(os.environ.get("ARENA_DATA_DIR", "data"))
con = duckdb.connect()
for name in ("questions", "forecasts", "resolutions", "skips"):
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{(DATA / (name + '.parquet')).as_posix()}')")
con.sql("SELECT domain, count(*) AS questions FROM questions GROUP BY domain ORDER BY domain").df()

## Brier score per forecaster

Brier = (probability - outcome)^2, with probabilities clipped to [0.01, 0.99] as on the site.
Lower is better; always answering 50% scores 0.25.

In [ ]:
brier = con.sql('''
    SELECT f.forecaster_id,
           count(*) AS n,
           avg(power(least(greatest(f.probability, 0.01), 0.99) - r.outcome, 2)) AS brier
    FROM forecasts f JOIN resolutions r USING (question_id)
    GROUP BY f.forecaster_id
    ORDER BY brier
''').df()
brier

## Calibration with pandas

When a forecaster says about 70%, does YES happen about 70% of the time? Forecasts are put into
five bins and the observed YES rate is compared with the mean forecast.

In [ ]:
scored = con.sql('''
    SELECT f.forecaster_id, f.probability, r.outcome
    FROM forecasts f JOIN resolutions r USING (question_id)
''').df()
scored["bin"] = pd.cut(scored["probability"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True)
calibration = (scored.groupby(["forecaster_id", "bin"], observed=True)
               .agg(n=("outcome", "size"), mean_forecast=("probability", "mean"),
                    yes_rate=("outcome", "mean"))
               .reset_index())
calibration

## Why forecasts were skipped

A skip is recorded instead of a guess; nobody fills in 50%.

In [ ]:
con.sql('''
    SELECT forecaster_id, reason_code, count(*) AS n
    FROM skips GROUP BY ALL ORDER BY forecaster_id, n DESC
''').df()

## The same question in Polars

The open data is plain Parquet, so any DataFrame library can read it. Here is the Brier score per forecaster again, computed lazily with Polars straight from the files, and checked against the DuckDB result above.


In [ ]:
import polars as pl

f = pl.scan_parquet(DATA / "forecasts.parquet")
r = pl.scan_parquet(DATA / "resolutions.parquet")
brier_pl = (
    f.join(r.select("question_id", "outcome"), on="question_id")
    .with_columns(p=pl.col("probability").clip(0.01, 0.99))
    .group_by("forecaster_id")
    .agg(n=pl.len(), brier=((pl.col("p") - pl.col("outcome")) ** 2).mean().round(4))
    .sort("brier")
    .collect()
)
brier_pl

In [ ]:
# Both engines must agree: the notebook fails if Polars and DuckDB disagree.
duck = dict(zip(brier["forecaster_id"], brier["brier"]))
assert set(duck) == set(brier_pl["forecaster_id"])
for fid, b in zip(brier_pl["forecaster_id"], brier_pl["brier"]):
    assert abs(duck[fid] - b) < 1e-3, (fid, duck[fid], b)
print("Polars and DuckDB agree on", len(duck), "forecasters")

In [ ]:
# A sanity check the CI run relies on: every scored probability is a valid probability.
assert ((scored["probability"] >= 0) & (scored["probability"] <= 1)).all()
assert set(scored["outcome"].unique()) <= {0, 1}
print(f"{len(scored)} scored forecasts, {brier['forecaster_id'].nunique()} forecasters")